In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from fetch_separate_datasets import fetch_data

In [2]:
(
    X, X_pre, X_post, X_combo, 
    Y, Y_exclude_anomalous, 
    onehot_cols, scale_cols, scale_cols_pre, scale_cols_post
) = fetch_data()

### Confirm that SMOTE correctly handles target sample imbalances

In [3]:
## Build the datasets for the 3 models

# split the data into training and test sets
X_train_pre, X_test_pre, y_train_pre, y_test_pre = train_test_split(X_pre, Y, test_size=0.2, random_state=42)
X_train_post, X_test_post, y_train_post, y_test_post = train_test_split(X_post, Y_exclude_anomalous, test_size=0.2, random_state=42)
X_train_combo, X_test_combo, y_train_combo, y_test_combo = train_test_split(X_combo, Y_exclude_anomalous, test_size=0.2, random_state=42)

# Transform the data 
column_transformer = ColumnTransformer(transformers=[
    ('onehot', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False), onehot_cols),
    ('scale', StandardScaler(), scale_cols_pre)
])
recoded_data_pre = column_transformer.fit_transform(X_train_pre)
feature_names = column_transformer.get_feature_names_out()
X_train_pre = pd.DataFrame(recoded_data_pre, columns=feature_names)
X_test_pre = pd.DataFrame(column_transformer.transform(X_test_pre), columns=feature_names)

column_transformer = ColumnTransformer(transformers=[
    ('scale', StandardScaler(), scale_cols_post)
])
recoded_data_post = column_transformer.fit_transform(X_train_post)
feature_names = column_transformer.get_feature_names_out()
X_train_post = pd.DataFrame(recoded_data_post, columns=feature_names)
X_test_post = pd.DataFrame(column_transformer.transform(X_test_post), columns=feature_names)

column_transformer = ColumnTransformer(transformers=[
    ('onehot', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False), onehot_cols),
    ('scale', StandardScaler(), scale_cols)
])
recoded_data_combo = column_transformer.fit_transform(X_train_combo)
feature_names = column_transformer.get_feature_names_out()
X_train_combo = pd.DataFrame(recoded_data_combo, columns=feature_names)
X_test_combo = pd.DataFrame(column_transformer.transform(X_test_combo), columns=feature_names)

datasets = [
    {'Xtrain': X_train_pre, 'Xtest': X_test_pre, 'Ytrain': y_train_pre, 'Ytest': y_test_pre},
    {'Xtrain': X_train_post, 'Xtest': X_test_post, 'Ytrain': y_train_post, 'Ytest': y_test_post},
    {'Xtrain': X_train_combo, 'Xtest': X_test_combo, 'Ytrain': y_train_combo, 'Ytest': y_test_combo},
]

c:\Users\Sturdy\anaconda3\envs\uciml\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [3, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\Sturdy\anaconda3\envs\uciml\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [1, 3, 4, 5] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [4]:
## Apply SMOTE to handle target class imbalance

smote = SMOTE(random_state=42)
for dataset in datasets:
    dataset['Xtrain'], dataset['Ytrain'] = smote.fit_resample(dataset['Xtrain'], dataset['Ytrain'])

c:\Users\Sturdy\anaconda3\envs\uciml\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
c:\Users\Sturdy\anaconda3\envs\uciml\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
c:\Users\Sturdy\anaconda3\envs\uciml\lib\site-packages\sklearn\base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


In [5]:
## Confirm each dataset has balanced sample counts
for i, dataset in enumerate(datasets):
    print(f"Dataset {i+1}:", dataset['Ytrain'].value_counts(), '\n')

Dataset 1: Target  
Dropout     1791
Enrolled    1791
Graduate    1791
Name: count, dtype: int64 

Dataset 2: Target  
Dropout     1708
Enrolled    1708
Graduate    1708
Name: count, dtype: int64 

Dataset 3: Target  
Dropout     1708
Enrolled    1708
Graduate    1708
Name: count, dtype: int64 

